# BirdCLEF 2026 — Custom ProtoSSM Training (Vertex AI / GPU)

Trains a **custom ProtoSSM** model on top of frozen Perch v2 embeddings.
Produces 5-fold weight files (`protossm_fold{i}.pt`) to be uploaded as a
Kaggle Dataset and loaded as an extra stream in the EoS-9 blend.

## Pipeline
1. Extract Perch v2 embeddings + logits from `train_audio/` and `train_soundscapes/` (cached)
2. Train ProtoSSM (d_model=320, 4 BiSSM layers, cross-attention) — 5-fold GroupKFold by site
3. Train ResidualSSM (second-pass correction) on top of ProtoSSM output
4. Report OOF macro-AUC per fold
5. Save weights + training config JSON

## Expected runtime on T4
- Embedding extraction: ~45–90 min (one-time, cached)
- ProtoSSM training (5 folds): ~2–3h total
- ResidualSSM training (5 folds): ~30 min total
- **Total: ~3–5h on T4**

## Output
- `outputs/protossm_fold{0..4}.pt`
- `outputs/residual_ssm_fold{0..4}.pt`
- `outputs/protossm_config.json` (architecture + label mapping)
- `outputs/oof_auc.json` (per-fold validation AUC)

Upload the `outputs/` directory as a Kaggle Dataset, then add it to EoS-9.

## Cell 0 — Install dependencies

In [ ]:
# Kaggle API (to download competition data if not already present)
# !pip install -q kaggle

# Core deps
!pip install -q onnxruntime soundfile librosa tqdm scikit-learn

# TensorFlow for Perch SavedModel (fallback if ONNX not available)
# !pip install -q tensorflow-cpu==2.20

print('✅ Dependencies ready')

## Cell 1 — Config

In [ ]:
from pathlib import Path
import os

# ── Paths ─────────────────────────────────────────────────────────────────────
# Adjust DATA_ROOT to wherever you've extracted the competition data.
# Expected layout:
#   DATA_ROOT/
#     taxonomy.csv
#     train.csv
#     train_soundscapes_labels.csv
#     train_audio/<species>/<filename>.ogg
#     train_soundscapes/<filename>.ogg
DATA_ROOT        = Path(os.environ.get('BIRDCLEF_DATA', '/home/jupyter/birdclef-2026/data/competition/raw'))
PERCH_ONNX_PATH  = Path(os.environ.get('PERCH_ONNX', '/home/jupyter/models/perch_v2.onnx'))
OUTPUT_DIR       = Path('outputs')
CACHE_DIR        = Path('cache')   # Perch embedding cache (large, ~10–20 GB)

OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

# ── Audio config ──────────────────────────────────────────────────────────────
SR             = 32_000
WINDOW_SEC     = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES   = 60 * SR
N_WINDOWS      = 12          # 60s / 5s
N_CLASSES      = 234
PERCH_N_LOGITS = 14795  # Perch v2 full label space

# ── Training config (V18 — best known public config) ─────────────────────────
PROTO_CFG = {
    'd_model': 320,
    'd_state': 32,
    'n_ssm_layers': 4,
    'dropout': 0.12,
    'n_prototypes': 2,
    'n_sites': 50,       # max sites in train set (use 50 to be safe)
    'meta_dim': 24,
    'use_cross_attn': True,
    'cross_attn_heads': 8,
}

TRAIN_CFG = {
    'n_epochs': 80,
    'lr': 8e-4,
    'weight_decay': 1e-3,
    'patience': 20,
    'pos_weight_cap': 25.0,
    'distill_weight': 0.15,
    'label_smoothing': 0.03,
    'n_folds': 5,
    'mixup_alpha': 0.4,
    'focal_gamma': 2.5,
    'swa_start_frac': 0.65,
    'swa_lr': 4e-4,
    'use_cosine_restart': True,
    'restart_period': 20,
}

RESIDUAL_CFG = {
    'd_model': 128,
    'd_state': 16,
    'n_ssm_layers': 2,
    'dropout': 0.1,
    'correction_weight': 0.35,
    'n_epochs': 40,
    'lr': 8e-4,
    'patience': 12,
}

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Data root: {DATA_ROOT} (exists={DATA_ROOT.exists()})')
print(f'Perch ONNX: {PERCH_ONNX_PATH} (exists={PERCH_ONNX_PATH.exists()})')

## Cell 2 — Imports & helpers

In [ ]:
import gc
import json
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

def macro_auc(y_true, y_score):
    """Competition metric: macro ROC-AUC, skip classes with no positives."""
    keep = y_true.sum(axis=0) > 0
    if keep.sum() == 0:
        return float('nan')
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average='macro')

FNAME_RE = re.compile(r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg')

def parse_fname(name):
    m = FNAME_RE.match(Path(name).name)
    if not m:
        return {'site': 'unknown', 'hour': 0}
    return {'site': m.group(2), 'hour': int(m.group(4)[:2])}

print('✅ Imports ready')

## Cell 3 — Load labels & taxonomy

In [ ]:
taxonomy = pd.read_csv(DATA_ROOT / 'taxonomy.csv')
PRIMARY_LABELS = taxonomy['primary_label'].tolist()
label2idx = {l: i for i, l in enumerate(PRIMARY_LABELS)}
assert len(PRIMARY_LABELS) == N_CLASSES, f'Expected {N_CLASSES} classes, got {len(PRIMARY_LABELS)}'

# Soundscape labels (the labeled portion of train_soundscapes)
ss_labels = pd.read_csv(DATA_ROOT / 'train_soundscapes_labels.csv')

def parse_time(t):
    parts = str(t).split(':')
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])

ss_labels['start_sec'] = ss_labels['start'].apply(parse_time)
ss_labels['end_sec']   = ss_labels['end'].apply(parse_time)

print(f'Taxonomy: {len(PRIMARY_LABELS)} classes')
print(f'Soundscape label rows: {len(ss_labels)}')
print(f'Soundscape files: {ss_labels["filename"].nunique()}')
# ── Perch label → competition label mapping (for knowledge distillation) ──────
PERCH_LABELS_PATH = Path('/home/jupyter/models/labels.csv')
if PERCH_LABELS_PATH.exists():
    perch_labels_df = pd.read_csv(PERCH_LABELS_PATH)
    # Column name varies: try 'inat2024_fsd50k' or first column
    col = 'inat2024_fsd50k' if 'inat2024_fsd50k' in perch_labels_df.columns else perch_labels_df.columns[0]
    perch_sci_names = perch_labels_df[col].tolist()
    perch_sci2idx = {n: i for i, n in enumerate(perch_sci_names)}
    NO_LABEL_INDEX = len(perch_sci_names)
    # Map competition labels → Perch indices
    BC_INDICES = np.array([
        perch_sci2idx.get(row['scientific_name'], NO_LABEL_INDEX)
        for _, row in taxonomy.iterrows()
    ], dtype=np.int32)
    MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
    MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
    MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)
    USE_DISTILL = True
    print(f'Perch mapping: {MAPPED_MASK.sum()}/{N_CLASSES} classes mapped')
else:
    USE_DISTILL = False
    MAPPED_POS = np.array([], dtype=np.int32)
    MAPPED_BC_INDICES = np.array([], dtype=np.int32)
    print('Perch labels.csv not found — distillation disabled')


## Cell 4 — Load Perch ONNX model

In [ ]:
import onnxruntime as ort

# Perch ONNX runs on CPU (the model was exported for CPU inference)
sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = os.cpu_count()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

perch_sess = ort.InferenceSession(
    str(PERCH_ONNX_PATH),
    sess_options=sess_options,
    providers=['CPUExecutionProvider'],
)

# Verify input/output names
print('Perch inputs: ', [i.name for i in perch_sess.get_inputs()])
print('Perch outputs:', [o.name for o in perch_sess.get_outputs()])

# Build output name → index map
_perch_out_map = {o.name: i for i, o in enumerate(perch_sess.get_outputs())}
_perch_in_name = perch_sess.get_inputs()[0].name

def run_perch(audio_window: np.ndarray):
    """Run Perch on a single 5-second window (float32, 32kHz).
    Returns (embedding [1536], perch_logits [PERCH_N_LOGITS])."""
    inp = audio_window.astype(np.float32)[None, :]   # (1, 160000)
    outs = perch_sess.run(None, {_perch_in_name: inp})
    emb    = outs[_perch_out_map['embedding']][0]     # (1536,)
    logits = outs[_perch_out_map['label']][0]          # (14795,)
    return emb.astype(np.float32), logits.astype(np.float32)

# Quick sanity check
_test = np.zeros(WINDOW_SAMPLES, dtype=np.float32)
_emb, _logit = run_perch(_test)
print(f'Perch emb shape: {_emb.shape}, logit shape: {_logit.shape}')  # expect (1536,) and (14795,)
print('✅ Perch ONNX loaded')

## Cell 5 — Extract Perch embeddings from train_soundscapes (labeled set)

We train on the **labeled soundscape windows** — these have ground-truth labels at the
5-second window level, so they're the ideal training signal for the SSM.

Cache is saved per-file to disk; re-running resumes from where it left off.

In [ ]:
TRAIN_SS_DIR = DATA_ROOT / 'train_soundscapes'

labeled_files = sorted(ss_labels['filename'].unique())
print(f'Labeled soundscape files: {len(labeled_files)}')

all_embeddings = []   # list of (n_windows, 1536)
all_logits     = []   # list of (n_windows, PERCH_N_LOGITS)
all_labels     = []   # list of (n_windows, N_CLASSES) binary
all_meta       = []   # list of dicts with site, hour, filename

for fname in tqdm(labeled_files, desc='Extracting Perch embeddings'):
    cache_file = CACHE_DIR / f'{Path(fname).stem}.npz'

    if cache_file.exists():
        cached = np.load(cache_file)
        file_emb    = cached['emb']
        file_logits = cached['logits']
    else:
        audio_path = TRAIN_SS_DIR / fname
        if not audio_path.exists():
            continue

        wave, orig_sr = sf.read(str(audio_path), dtype='float32')
        if wave.ndim > 1:
            wave = wave.mean(axis=1)
        if orig_sr != SR:
            import librosa
            wave = librosa.resample(wave, orig_sr=orig_sr, target_sr=SR)

        # Pad or truncate to exactly 60 seconds
        if len(wave) < FILE_SAMPLES:
            wave = np.pad(wave, (0, FILE_SAMPLES - len(wave)))
        else:
            wave = wave[:FILE_SAMPLES]

        file_emb    = np.zeros((N_WINDOWS, 1536), dtype=np.float32)
        file_logits = np.zeros((N_WINDOWS, PERCH_N_LOGITS), dtype=np.float32)

        for w in range(N_WINDOWS):
            start = w * WINDOW_SAMPLES
            window = wave[start: start + WINDOW_SAMPLES]
            file_emb[w], file_logits[w] = run_perch(window)

        np.savez_compressed(str(cache_file), emb=file_emb, logits=file_logits)

    # Build per-window labels from soundscape annotations
    file_rows = ss_labels[ss_labels['filename'] == fname]
    file_label_matrix = np.zeros((N_WINDOWS, N_CLASSES), dtype=np.float32)

    for _, row in file_rows.iterrows():
        if row['primary_label'] not in label2idx:
            continue
        cls_idx = label2idx[row['primary_label']]
        # Mark all 5-second windows that overlap with the annotation
        for w in range(N_WINDOWS):
            win_start = w * WINDOW_SEC
            win_end   = win_start + WINDOW_SEC
            if row['start_sec'] < win_end and row['end_sec'] > win_start:
                file_label_matrix[w, cls_idx] = 1.0

    meta = parse_fname(fname)
    meta['filename'] = fname

    all_embeddings.append(file_emb)
    all_logits.append(file_logits)
    all_labels.append(file_label_matrix)
    all_meta.append(meta)

print(f'Total labeled files extracted: {len(all_embeddings)}')

# Stack: (N_files, N_windows, dim)
EMB_ALL    = np.stack(all_embeddings)   # (F, 12, 1536)
_raw_logits = np.stack(all_logits)      # (F, 12, PERCH_N_LOGITS)
LABELS_ALL = np.stack(all_labels)       # (F, 12, N_CLASSES)

# Map raw Perch logits (14795) → competition class space (234) via sigmoid for soft labels
LOGITS_ALL = np.zeros((_raw_logits.shape[0], _raw_logits.shape[1], N_CLASSES), dtype=np.float32)
if USE_DISTILL and len(MAPPED_POS) > 0:
    # sigmoid to convert logits → probabilities for distillation targets
    LOGITS_ALL[:, :, MAPPED_POS] = 1.0 / (1.0 + np.exp(-_raw_logits[:, :, MAPPED_BC_INDICES]))
    print(f'Distillation targets built: {MAPPED_POS.shape[0]}/{N_CLASSES} classes have Perch soft labels')
del _raw_logits
META_ALL   = pd.DataFrame(all_meta)

# Encode sites/hours as integers
unique_sites = sorted(META_ALL['site'].unique())
site2idx = {s: i for i, s in enumerate(unique_sites)}
META_ALL['site_idx'] = META_ALL['site'].map(site2idx)
META_ALL['hour_idx'] = META_ALL['hour'].clip(0, 23)

# Update PROTO_CFG with actual number of sites
PROTO_CFG['n_sites'] = max(len(unique_sites) + 1, 50)

print(f'Shape: emb={EMB_ALL.shape}, logits={LOGITS_ALL.shape}, labels={LABELS_ALL.shape}')
print(f'Sites: {len(unique_sites)}, Hours: 0-23')
print(f'Positive windows: {LABELS_ALL.any(axis=-1).mean()*100:.1f}%')

## Cell 6 — Model architecture (SelectiveSSM + ProtoSSMv2 + ResidualSSM)

In [ ]:
class SelectiveSSM(nn.Module):
    """Simplified Mamba-style selective state space model."""

    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        self.in_proj   = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d    = nn.Conv1d(d_model, d_model, d_conv,
                                   padding=d_conv - 1, groups=d_model)
        self.dt_proj   = nn.Linear(d_model, d_model, bias=True)

        A = torch.arange(1, d_state + 1, dtype=torch.float32)
        self.A_log     = nn.Parameter(torch.log(A.unsqueeze(0).expand(d_model, -1)))
        self.D         = nn.Parameter(torch.ones(d_model))
        self.B_proj    = nn.Linear(d_model, d_state, bias=False)
        self.C_proj    = nn.Linear(d_model, d_state, bias=False)
        self.out_proj  = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B_size, T, D = x.shape
        x_ssm, z = self.in_proj(x).chunk(2, dim=-1)

        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)

        dt = F.softplus(self.dt_proj(x_conv))
        A  = -torch.exp(self.A_log)          # (d_model, d_state)
        B  = self.B_proj(x_conv)             # (B, T, d_state)
        C  = self.C_proj(x_conv)             # (B, T, d_state)

        h   = torch.zeros(B_size, D, self.d_state, device=x.device)
        ys  = []
        for t in range(T):
            dt_t = dt[:, t, :]
            dA   = torch.exp(A[None, :, :] * dt_t[:, :, None])
            dB   = dt_t[:, :, None] * B[:, t, None, :]
            h    = h * dA + x[:, t, :, None] * dB
            ys.append((h * C[:, t, None, :]).sum(-1))

        return torch.stack(ys, dim=1) + x * self.D[None, None, :]


class TemporalCrossAttention(nn.Module):

    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm  = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model), nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        residual = x
        attn_out, _ = self.attn(self.norm(x), self.norm(x), self.norm(x))
        x = residual + attn_out
        return x + self.ffn(self.norm2(x))


class SSMBlock(nn.Module):
    """One bidirectional SSM layer with residual and norm."""

    def __init__(self, d_model, d_state=16, dropout=0.1):
        super().__init__()
        self.ssm_fwd  = SelectiveSSM(d_model, d_state)
        self.ssm_bwd  = SelectiveSSM(d_model, d_state)
        self.merge    = nn.Linear(2 * d_model, d_model)
        self.norm     = nn.LayerNorm(d_model)
        self.drop     = nn.Dropout(dropout)

    def forward(self, x):
        h_f = self.ssm_fwd(x)
        h_b = self.ssm_bwd(x.flip(1)).flip(1)
        h   = self.drop(self.merge(torch.cat([h_f, h_b], dim=-1)))
        return self.norm(h + x)


class ProtoSSMv2(nn.Module):
    """ProtoSSM v4 — 4-layer bidirectional SSM with cross-attention and gated fusion."""

    def __init__(self, cfg, n_classes=234, n_windows=12, perch_dim=1536):
        super().__init__()
        d_model  = cfg['d_model']
        d_state  = cfg['d_state']
        n_layers = cfg['n_ssm_layers']
        dropout  = cfg['dropout']
        n_sites  = cfg['n_sites']
        meta_dim = cfg['meta_dim']

        # Input projection: Perch embedding → d_model
        self.input_proj = nn.Sequential(
            nn.Linear(perch_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Site + hour metadata embeddings
        self.site_emb = nn.Embedding(n_sites + 1, meta_dim, padding_idx=0)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)

        # Positional encoding
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)

        # N bidirectional SSM layers
        self.ssm_layers = nn.ModuleList([
            SSMBlock(d_model, d_state, dropout) for _ in range(n_layers)
        ])

        # Optional cross-attention after SSM layers
        self.use_cross_attn = cfg.get('use_cross_attn', False)
        if self.use_cross_attn:
            self.cross_attn = TemporalCrossAttention(
                d_model, cfg.get('cross_attn_heads', 4), dropout)

        # Gated fusion: blend SSM output with Perch logits
        self.gate_alpha = nn.Parameter(torch.full((n_classes,), 0.5))
        self.class_bias = nn.Parameter(torch.zeros(n_classes))

        # Output head
        self.output_head = nn.Linear(d_model, n_classes)

    def forward(self, emb, perch_logits, site_ids=None, hours=None):
        # emb: (B, T, 1536), perch_logits: (B, T, n_classes)
        h = self.input_proj(emb)                     # (B, T, d_model)

        if site_ids is not None and hours is not None:
            s = self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1))
            o = self.hour_emb(hours.clamp(0, 23))
            h = h + self.meta_proj(torch.cat([s, o], dim=-1)).unsqueeze(1)

        h = h + self.pos_enc[:, :emb.shape[1], :]

        for layer in self.ssm_layers:
            h = layer(h)

        if self.use_cross_attn:
            h = self.cross_attn(h)

        ssm_logits   = self.output_head(h)                      # (B, T, n_classes)
        alpha        = torch.sigmoid(self.gate_alpha)           # (n_classes,)
        fused_logits = alpha * ssm_logits + (1 - alpha) * perch_logits + self.class_bias
        return fused_logits


class ResidualSSM(nn.Module):
    """Second-pass correction: takes ProtoSSM output + embeddings → additive correction."""

    def __init__(self, cfg, n_classes=234, n_windows=12, perch_dim=1536):
        super().__init__()
        d_model  = cfg['d_model']
        d_state  = cfg['d_state']
        dropout  = cfg['dropout']
        n_sites  = PROTO_CFG['n_sites']
        meta_dim = 8

        self.input_proj = nn.Sequential(
            nn.Linear(perch_dim + n_classes, d_model),
            nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout),
        )
        self.site_emb  = nn.Embedding(n_sites + 1, meta_dim, padding_idx=0)
        self.hour_emb  = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.pos_enc   = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)

        self.ssm_layers = nn.ModuleList([
            SSMBlock(d_model, d_state, dropout) for _ in range(cfg['n_ssm_layers'])
        ])

        self.output_head = nn.Linear(d_model, n_classes)
        nn.init.zeros_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)

    def forward(self, emb, first_pass_logits, site_ids=None, hours=None):
        x = torch.cat([emb, first_pass_logits], dim=-1)
        h = self.input_proj(x)
        if site_ids is not None and hours is not None:
            s = self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1))
            o = self.hour_emb(hours.clamp(0, 23))
            h = h + self.meta_proj(torch.cat([s, o], dim=-1)).unsqueeze(1)
        h = h + self.pos_enc[:, :emb.shape[1], :]
        for layer in self.ssm_layers:
            h = layer(h)
        return self.output_head(h)   # additive correction (B, T, n_classes)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Quick size check
_proto = ProtoSSMv2(PROTO_CFG, N_CLASSES, N_WINDOWS)
_resid = ResidualSSM(RESIDUAL_CFG, N_CLASSES, N_WINDOWS)
print(f'ProtoSSMv2 params: {count_params(_proto):,}')
print(f'ResidualSSM params: {count_params(_resid):,}')
del _proto, _resid
print('✅ Model classes defined')

## Cell 7 — Training loop

In [ ]:
from torch.optim import AdamW
from torch.optim.swa_utils import AveragedModel, SWALR


def focal_bce_loss(logits, targets, gamma=2.0, pos_weight=None):
    """Focal binary cross-entropy for multi-label classification."""
    bce = F.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=pos_weight, reduction='none')
    p_t = torch.exp(-bce)
    return ((1 - p_t) ** gamma * bce).mean()


def mixup_batch(emb, logits, labels, alpha=0.4):
    """File-level mixup: blend two files' embeddings + labels."""
    if alpha <= 0:
        return emb, logits, labels
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(emb.shape[0], device=emb.device)
    return (lam * emb + (1 - lam) * emb[idx],
            lam * logits + (1 - lam) * logits[idx],
            torch.clamp(labels + labels[idx], 0, 1))   # union of labels


def train_fold(proto_model, emb_tr, logits_tr, labels_tr, site_tr, hour_tr,
               emb_val, logits_val, labels_val, site_val, hour_val,
               cfg, fold_idx):
    """Train one fold of ProtoSSM. Returns (best_model_state, oof_auc)."""
    device = DEVICE
    proto_model = proto_model.to(device)

    # Tensors
    E_tr  = torch.tensor(emb_tr,    dtype=torch.float32, device=device)
    L_tr  = torch.tensor(logits_tr, dtype=torch.float32, device=device)
    Y_tr  = torch.tensor(labels_tr, dtype=torch.float32, device=device)
    S_tr  = torch.tensor(site_tr,   dtype=torch.long,    device=device)
    H_tr  = torch.tensor(hour_tr,   dtype=torch.long,    device=device)

    E_val = torch.tensor(emb_val,    dtype=torch.float32, device=device)
    L_val = torch.tensor(logits_val, dtype=torch.float32, device=device)
    Y_val = torch.tensor(labels_val, dtype=torch.float32, device=device)
    S_val = torch.tensor(site_val,   dtype=torch.long,    device=device)
    H_val = torch.tensor(hour_val,   dtype=torch.long,    device=device)

    # Class pos weights (capped)
    pos_counts = Y_tr.sum(dim=(0, 1)).clamp(1)
    neg_counts = (1 - Y_tr).sum(dim=(0, 1)).clamp(1)
    pos_weight = (neg_counts / pos_counts).clamp(max=cfg['pos_weight_cap'])

    optimizer = AdamW(proto_model.parameters(), lr=cfg['lr'],
                      weight_decay=cfg['weight_decay'])

    if cfg.get('use_cosine_restart'):
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=cfg['restart_period'], eta_min=1e-5)
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg['n_epochs'], eta_min=1e-5)

    n_epochs      = cfg['n_epochs']
    swa_start     = int(n_epochs * cfg['swa_start_frac'])
    swa_model     = AveragedModel(proto_model)
    swa_scheduler = SWALR(optimizer, swa_lr=cfg['swa_lr'])
    using_swa     = False

    label_smoothing = cfg.get('label_smoothing', 0.0)

    best_auc    = -1.0
    best_state  = None
    patience_ct = 0
    history     = []

    BATCH = 32
    n_files = E_tr.shape[0]

    for epoch in range(1, n_epochs + 1):
        proto_model.train()
        perm = torch.randperm(n_files, device=device)
        epoch_loss = 0.0
        n_batches  = 0

        for start in range(0, n_files, BATCH):
            idx = perm[start: start + BATCH]
            e, l, y = E_tr[idx], L_tr[idx], Y_tr[idx]
            s, h    = S_tr[idx], H_tr[idx]

            if cfg.get('mixup_alpha', 0) > 0 and np.random.rand() < 0.5:
                e, l, y = mixup_batch(e, l, y, cfg['mixup_alpha'])

            if label_smoothing > 0:
                y = y * (1 - label_smoothing) + label_smoothing / 2

            optimizer.zero_grad()
            pred_logits = proto_model(e, l, s, h)
            # Flatten (B, T, C) → (B*T, C) for loss
            loss = focal_bce_loss(
                pred_logits.reshape(-1, N_CLASSES),
                y.reshape(-1, N_CLASSES),
                gamma=cfg.get('focal_gamma', 2.0),
                pos_weight=pos_weight,
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(proto_model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
            n_batches  += 1

        if epoch >= swa_start:
            swa_model.update_parameters(proto_model)
            swa_scheduler.step()
            using_swa = True
        else:
            scheduler.step()

        # Validation
        eval_model = swa_model if using_swa else proto_model
        eval_model.eval()
        with torch.no_grad():
            val_logits = eval_model(E_val, L_val, S_val, H_val)
            val_probs  = torch.sigmoid(val_logits).cpu().numpy()

        y_val_flat  = Y_val.cpu().numpy().reshape(-1, N_CLASSES)
        val_probs_f = val_probs.reshape(-1, N_CLASSES)
        val_auc     = macro_auc(y_val_flat, val_probs_f)

        avg_loss = epoch_loss / max(n_batches, 1)
        history.append({'epoch': epoch, 'loss': avg_loss, 'val_auc': val_auc})

        if epoch % 10 == 0 or epoch == n_epochs:
            print(f'  Fold {fold_idx} | Epoch {epoch:3d}/{n_epochs} | '
                  f'loss={avg_loss:.4f} | val_auc={val_auc:.4f} | '
                  f'{"SWA" if using_swa else "   "}')

        if val_auc > best_auc:
            best_auc   = val_auc
            eval_model.eval()
            # Save the state dict of the eval model (SWA or regular)
            best_state = {k: v.cpu().clone()
                          for k, v in eval_model.state_dict().items()}
            patience_ct = 0
        else:
            patience_ct += 1
            if patience_ct >= cfg['patience']:
                print(f'  Early stop at epoch {epoch}')
                break

    torch.cuda.empty_cache()
    return best_state, best_auc, history


print('✅ Training loop defined')

## Cell 8 — 5-fold cross-validation training

In [ ]:
from sklearn.model_selection import GroupKFold

N_FOLDS     = TRAIN_CFG['n_folds']
gkf         = GroupKFold(n_splits=N_FOLDS)
groups      = META_ALL['site'].values    # split by recording site
indices     = np.arange(len(META_ALL))

fold_results = []
oof_preds    = np.zeros_like(LABELS_ALL[:, :, :].reshape(-1, N_CLASSES), dtype=np.float32)
oof_labels   = LABELS_ALL.reshape(-1, N_CLASSES)
oof_idx_map  = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(indices, groups=groups)):
    print(f'\n{'='*60}')
    print(f'FOLD {fold+1}/{N_FOLDS} | train={len(tr_idx)} | val={len(val_idx)}')
    print(f'{'='*60}')

    emb_tr   = EMB_ALL[tr_idx]
    emb_val  = EMB_ALL[val_idx]
    log_tr   = LOGITS_ALL[tr_idx]
    log_val  = LOGITS_ALL[val_idx]
    lab_tr   = LABELS_ALL[tr_idx]
    lab_val  = LABELS_ALL[val_idx]
    site_tr  = META_ALL['site_idx'].values[tr_idx]
    site_val = META_ALL['site_idx'].values[val_idx]
    hour_tr  = META_ALL['hour_idx'].values[tr_idx]
    hour_val = META_ALL['hour_idx'].values[val_idx]

    proto_model = ProtoSSMv2(PROTO_CFG, N_CLASSES, N_WINDOWS)

    best_state, best_auc, history = train_fold(
        proto_model,
        emb_tr, log_tr, lab_tr, site_tr, hour_tr,
        emb_val, log_val, lab_val, site_val, hour_val,
        TRAIN_CFG, fold_idx=fold + 1,
    )

    print(f'\n  ✅ Fold {fold+1} best val AUC: {best_auc:.4f}')

    # Save weights
    torch.save(best_state, OUTPUT_DIR / f'protossm_fold{fold}.pt')

    # OOF predictions
    proto_model.load_state_dict(best_state)
    proto_model.eval().to(DEVICE)
    with torch.no_grad():
        val_logits = proto_model(
            torch.tensor(emb_val, dtype=torch.float32, device=DEVICE),
            torch.tensor(log_val, dtype=torch.float32, device=DEVICE),
            torch.tensor(site_val, dtype=torch.long, device=DEVICE),
            torch.tensor(hour_val, dtype=torch.long, device=DEVICE),
        )
        val_probs = torch.sigmoid(val_logits).cpu().numpy()

    flat_val_idx = []
    for fi in val_idx:
        for w in range(N_WINDOWS):
            flat_val_idx.append(fi * N_WINDOWS + w)
    oof_preds[flat_val_idx] = val_probs.reshape(-1, N_CLASSES)
    oof_idx_map.extend(flat_val_idx)

    fold_results.append({'fold': fold + 1, 'val_auc': best_auc, 'history': history})

    del proto_model
    torch.cuda.empty_cache()
    gc.collect()

# OOF summary
oof_auc = macro_auc(oof_labels[oof_idx_map], oof_preds[oof_idx_map])
fold_aucs = [r['val_auc'] for r in fold_results]
print(f'\n{"="*60}')
print(f'OOF AUC: {oof_auc:.4f}')
print(f'Per-fold: {[f"{a:.4f}" for a in fold_aucs]}')
print(f'Mean:     {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}')
print(f'='*60)

## Cell 9 — ResidualSSM second-pass training

In [ ]:
# Train ResidualSSM using the ProtoSSM outputs as input

for fold, (tr_idx, val_idx) in enumerate(gkf.split(indices, groups=groups)):
    print(f'\nResidualSSM Fold {fold+1}/{N_FOLDS}')

    # Load ProtoSSM weights for this fold
    proto_model = ProtoSSMv2(PROTO_CFG, N_CLASSES, N_WINDOWS).to(DEVICE)
    proto_model.load_state_dict(torch.load(OUTPUT_DIR / f'protossm_fold{fold}.pt',
                                            map_location=DEVICE))
    proto_model.eval()

    # Generate ProtoSSM predictions for train/val
    with torch.no_grad():
        for split_name, split_idx in [('train', tr_idx), ('val', val_idx)]:
            e = torch.tensor(EMB_ALL[split_idx],    dtype=torch.float32, device=DEVICE)
            l = torch.tensor(LOGITS_ALL[split_idx], dtype=torch.float32, device=DEVICE)
            s = torch.tensor(META_ALL['site_idx'].values[split_idx], dtype=torch.long, device=DEVICE)
            h = torch.tensor(META_ALL['hour_idx'].values[split_idx], dtype=torch.long, device=DEVICE)
            proto_out = proto_model(e, l, s, h)  # (N, T, C) logits
            if split_name == 'train':
                proto_tr = proto_out.cpu()
            else:
                proto_val = proto_out.cpu()

    # Train ResidualSSM
    res_model = ResidualSSM(RESIDUAL_CFG, N_CLASSES, N_WINDOWS).to(DEVICE)
    optimizer = AdamW(res_model.parameters(), lr=RESIDUAL_CFG['lr'], weight_decay=1e-4)

    E_tr_r  = torch.tensor(EMB_ALL[tr_idx],             dtype=torch.float32, device=DEVICE)
    Y_tr_r  = torch.tensor(LABELS_ALL[tr_idx],          dtype=torch.float32, device=DEVICE)
    S_tr_r  = torch.tensor(META_ALL['site_idx'].values[tr_idx], dtype=torch.long, device=DEVICE)
    H_tr_r  = torch.tensor(META_ALL['hour_idx'].values[tr_idx], dtype=torch.long, device=DEVICE)
    P_tr_r  = proto_tr.to(DEVICE)

    E_val_r = torch.tensor(EMB_ALL[val_idx],             dtype=torch.float32, device=DEVICE)
    Y_val_r = torch.tensor(LABELS_ALL[val_idx],          dtype=torch.float32, device=DEVICE)
    S_val_r = torch.tensor(META_ALL['site_idx'].values[val_idx], dtype=torch.long, device=DEVICE)
    H_val_r = torch.tensor(META_ALL['hour_idx'].values[val_idx], dtype=torch.long, device=DEVICE)
    P_val_r = proto_val.to(DEVICE)

    best_auc_r   = -1.0
    best_state_r = None
    patience_ct  = 0
    BATCH        = 32
    n_files      = len(tr_idx)
    n_epochs_r   = RESIDUAL_CFG['n_epochs']

    for epoch in range(1, n_epochs_r + 1):
        res_model.train()
        perm = torch.randperm(n_files, device=DEVICE)

        for start in range(0, n_files, BATCH):
            idx_b = perm[start: start + BATCH]
            optimizer.zero_grad()
            correction = res_model(E_tr_r[idx_b], P_tr_r[idx_b].detach(), S_tr_r[idx_b], H_tr_r[idx_b])
            final_logits = P_tr_r[idx_b] + correction * RESIDUAL_CFG['correction_weight']
            loss = focal_bce_loss(final_logits.reshape(-1, N_CLASSES), Y_tr_r[idx_b].reshape(-1, N_CLASSES))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(res_model.parameters(), 1.0)
            optimizer.step()

        if epoch % 10 == 0 or epoch == n_epochs_r:
            res_model.eval()
            with torch.no_grad():
                correction = res_model(E_val_r, P_val_r.detach(), S_val_r, H_val_r)
                final_logits = P_val_r + correction * RESIDUAL_CFG['correction_weight']
                val_probs = torch.sigmoid(final_logits).cpu().numpy()
            auc = macro_auc(LABELS_ALL[val_idx].reshape(-1, N_CLASSES),
                            val_probs.reshape(-1, N_CLASSES))
            print(f'  Fold {fold+1} Residual | Epoch {epoch:3d} | val_auc={auc:.4f}')

            if auc > best_auc_r:
                best_auc_r   = auc
                best_state_r = {k: v.cpu().clone() for k, v in res_model.state_dict().items()}
                patience_ct  = 0
            else:
                patience_ct += 1
                if patience_ct >= RESIDUAL_CFG['patience'] // 10:
                    break

    torch.save(best_state_r, OUTPUT_DIR / f'residual_ssm_fold{fold}.pt')
    print(f'  ✅ ResidualSSM Fold {fold+1} best AUC: {best_auc_r:.4f}')

    del proto_model, res_model, proto_tr, proto_val
    torch.cuda.empty_cache()
    gc.collect()

## Cell 10 — Save config + summary

In [ ]:
import json

# Save architecture config (needed to reconstruct model in inference)
config_out = {
    'proto_ssm': PROTO_CFG,
    'residual_ssm': RESIDUAL_CFG,
    'train_cfg': TRAIN_CFG,
    'n_classes': N_CLASSES,
    'n_windows': N_WINDOWS,
    'n_folds': N_FOLDS,
    'primary_labels': PRIMARY_LABELS,
    'site2idx': site2idx,
    'oof_auc': oof_auc,
    'fold_aucs': [r['val_auc'] for r in fold_results],
    'mean_auc': float(np.mean([r['val_auc'] for r in fold_results])),
}

with open(OUTPUT_DIR / 'protossm_config.json', 'w') as f:
    json.dump(config_out, f, indent=2)

# List saved files
saved = sorted(OUTPUT_DIR.iterdir())
print('\n📁 Saved to outputs/:')
for p in saved:
    size_mb = p.stat().st_size / 1e6
    print(f'  {p.name:40s}  {size_mb:.1f} MB')

print(f'\n✅ Training complete!')
print(f'   OOF AUC:  {oof_auc:.4f}')
print(f'   Mean AUC: {np.mean([r["val_auc"] for r in fold_results]):.4f}')
print(f'\n📌 Next step: upload outputs/ as a Kaggle Dataset,')
print(f'   then add protossm_fold*.pt as a new stream in EoS-9.')

## Cell 11 — (Optional) Upload to Kaggle as Dataset

Run this cell to push the trained weights directly to Kaggle.
Requires `kaggle.json` credentials in `~/.kaggle/`.

In [ ]:
KAGGLE_USERNAME = 'aleixlopez'     # ← your Kaggle username
DATASET_SLUG    = 'birdclef2026-custom-protossm-weights'

import json
from pathlib import Path

dataset_dir = Path('kaggle_upload')
dataset_dir.mkdir(exist_ok=True)

# Copy weights
import shutil
for f in sorted(OUTPUT_DIR.iterdir()):
    shutil.copy(f, dataset_dir / f.name)

# Write dataset-metadata.json
meta = {
    'title': 'BirdCLEF 2026 Custom ProtoSSM Weights',
    'id': f'{KAGGLE_USERNAME}/{DATASET_SLUG}',
    'licenses': [{'name': 'CC0-1.0'}],
}
with open(dataset_dir / 'dataset-metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

# Push to Kaggle
!kaggle datasets create -p {dataset_dir} --dir-mode zip
print(f'✅ Uploaded to kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_SLUG}')